# 🚗 ANPR System — YOLOv8 Training Notebook
### Automatic Number Plate Recognition | Google Colab

**Steps:**
1. Install dependencies
2. Upload & extract dataset
3. Convert XML → YOLO format
4. Create data.yaml
5. Train YOLOv8n
6. Download model


## ✅ Step 1: Install Dependencies

In [ ]:
!pip install ultralytics opencv-python-headless easyocr streamlit -q
print('✅ All packages installed!')

## ✅ Step 2: Upload Dataset from Kaggle

**Instructions:**
1. Go to: https://www.kaggle.com/datasets/andrewmvd/car-plate-detection
2. Download the dataset ZIP
3. Run the cell below and upload the ZIP file when prompted


In [ ]:
from google.colab import files
import zipfile, os

print('📁 Upload your dataset ZIP file...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/dataset_raw')

print('✅ Dataset extracted!')
!find /content/dataset_raw -name '*.xml' | head -5
!find /content/dataset_raw -name '*.png' -o -name '*.jpg' | head -5

## ✅ Step 3: Set Up Directory Structure

In [ ]:
import os
import shutil

# Create YOLO directory structure
dirs = [
    '/content/anpr_dataset/images/train',
    '/content/anpr_dataset/images/val',
    '/content/anpr_dataset/labels/train',
    '/content/anpr_dataset/labels/val',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print('✅ Directory structure created!')

## ✅ Step 4: Convert XML → YOLO Format & Split Dataset

In [ ]:
import xml.etree.ElementTree as ET
import glob
import random
import shutil

def convert_xml_to_yolo(xml_path):
    """Convert Pascal VOC XML to YOLO format."""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    img_w = int(root.find('size/width').text)
    img_h = int(root.find('size/height').text)

    yolo_lines = []
    for obj in root.findall('object'):
        # Class index 0 = license_plate
        class_id = 0
        bb = obj.find('bndbox')
        xmin = float(bb.find('xmin').text)
        ymin = float(bb.find('ymin').text)
        xmax = float(bb.find('xmax').text)
        ymax = float(bb.find('ymax').text)

        # Convert to YOLO normalized format
        x_center = ((xmin + xmax) / 2) / img_w
        y_center = ((ymin + ymax) / 2) / img_h
        width    = (xmax - xmin) / img_w
        height   = (ymax - ymin) / img_h

        yolo_lines.append(f'{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}')

    return yolo_lines


# Find all XML annotation files
xml_files = glob.glob('/content/dataset_raw/**/*.xml', recursive=True)
print(f'📄 Found {len(xml_files)} XML annotation files')

# Shuffle and split 80/20
random.seed(42)
random.shuffle(xml_files)
split = int(len(xml_files) * 0.8)
train_xmls = xml_files[:split]
val_xmls   = xml_files[split:]

def process_split(xml_list, split_name):
    count = 0
    for xml_path in xml_list:
        # Find matching image
        base = os.path.splitext(xml_path)[0]
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.PNG']:
            candidate = base + ext
            if os.path.exists(candidate):
                img_path = candidate
                break

        if img_path is None:
            # Search in same directory
            folder = os.path.dirname(xml_path)
            fname  = os.path.splitext(os.path.basename(xml_path))[0]
            for f in os.listdir(folder):
                if os.path.splitext(f)[0] == fname and f.endswith(('.jpg','.jpeg','.png')):
                    img_path = os.path.join(folder, f)
                    break

        if img_path is None:
            continue

        yolo_lines = convert_xml_to_yolo(xml_path)
        if not yolo_lines:
            continue

        fname = os.path.basename(img_path)
        label_name = os.path.splitext(fname)[0] + '.txt'

        shutil.copy(img_path, f'/content/anpr_dataset/images/{split_name}/{fname}')

        with open(f'/content/anpr_dataset/labels/{split_name}/{label_name}', 'w') as f:
            f.write('\n'.join(yolo_lines))

        count += 1
    return count

train_count = process_split(train_xmls, 'train')
val_count   = process_split(val_xmls,   'val')

print(f'✅ Train: {train_count} images | Val: {val_count} images')

## ✅ Step 5: Create data.yaml

In [ ]:
yaml_content = """
path: /content/anpr_dataset
train: images/train
val: images/val

nc: 1
names: ['license_plate']
"""

with open('/content/anpr_dataset/data.yaml', 'w') as f:
    f.write(yaml_content.strip())

print('✅ data.yaml created!')
!cat /content/anpr_dataset/data.yaml

## ✅ Step 6: Verify GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))

## ✅ Step 7: Train YOLOv8n (10 epochs, fast)

In [ ]:
from ultralytics import YOLO

# Load YOLOv8 nano — smallest and fastest
model = YOLO('yolov8n.pt')

# Train
results = model.train(
    data='/content/anpr_dataset/data.yaml',
    epochs=10,
    imgsz=640,
    batch=16,
    name='anpr_model',
    device=0,           # GPU
    patience=5,
    save=True,
    verbose=True
)

print('✅ Training complete!')

## ✅ Step 8: Validate Model

In [ ]:
# Load best weights
best_model = YOLO('/content/runs/detect/anpr_model/weights/best.pt')

# Validate on val set
metrics = best_model.val(data='/content/anpr_dataset/data.yaml')
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

## ✅ Step 9: Quick Test on a Validation Image

In [ ]:
import glob
from IPython.display import Image as IPImage, display

# Pick a validation image
val_images = glob.glob('/content/anpr_dataset/images/val/*')[:1]

if val_images:
    result = best_model.predict(val_images[0], save=True, project='/content/test_output', name='result')
    result_img = glob.glob('/content/test_output/result/*')[0]
    display(IPImage(result_img, width=600))
    print('✅ Detection test complete!')
else:
    print('No val images found')

## ✅ Step 10: Download the Trained Model

In [ ]:
from google.colab import files
import shutil

# Copy best.pt to a convenient location
shutil.copy(
    '/content/runs/detect/anpr_model/weights/best.pt',
    '/content/anpr_best.pt'
)

files.download('/content/anpr_best.pt')
print('✅ Model downloaded! Save it as anpr_best.pt in your project folder.')